## Issue 6 (Purav): Production ESMFold2 fold of the 90pid centroids

Lead     : `Purav Gupta`

Issue    : [Github Issue #6](https://github.com/petadex/igem-toronto/issues/6) — _fold the family-cluster centroids_

Start    : `2026-08-06`

Complete : `N/A`

Files    : `resources/260806_issue6_purav_foldrun/data/` — local working dir on Compute Canada `/project` (git-ignored)

S3 files : `s3://petadex-protein-structures/esmfold2-centroids/` — remote archive (synced after batches, not during)

---

### Data Accessed: 2026-06-21
Input is the 90pid catalytic-core FASTA already on disk (`90pid_catalytic_orfs.fa`, 18,172,960 records), derived earlier from SQL. That table was then saved at `s3://petabite/petadex/90pid_enzymes_clusters.csv`

```bash
cat 90pid_enzymes_clusters.csv | cut -f 2 -d ',' > 90_orfs.txt
aws s3 cp s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa - --no-sign-request \
  | seqkit grep --id-regexp '^([^|]+)' -f 90_orfs.txt > 90pid_catalytic_orfs.fa
```

### Data Accessed: 2026-10-15 (Part II)
- **PETadex HF corpora** (Dennis Zhu, built 2026-10-15, private): `petadex/petadex-corpus-centroids90` (`centroids90.parquet`, 18,172,960 rows, one per 90% cluster) and `petadex/petadex-corpus-full` (307,155,746 ORFs; `orfs/` spine plus `layers/` and `dims/`). Local copy at `/project/6122149/purav/petadex-corpus/`. The centroid IDs match `90_orfs.txt` exactly, so this is the same fold input.
- **RCSB PDB** (public GraphQL + `files.rcsb.org`), queried 2026-10-15 for release date, method and resolution of every PDB accession in the corpus.
- **AlphaFold DB structures** from `s3://petadex-protein-structures/af_db_v260701/structures/` (via the corpus `afdb_structures` layer), pulled 2026-10-15 with AWS CLI v2 SSO (`--profile igem-purav`).

```bash
# HF corpora (anonymous HTTPS; large layers are queried remotely with DuckDB instead)
B=https://huggingface.co/datasets/petadex
curl -L -o centroids90/centroids90.parquet $B/petadex-corpus-centroids90/resolve/main/centroids90.parquet
curl -L -o full/layers/afdb_structures.parquet $B/petadex-corpus-full/resolve/main/layers/afdb_structures.parquet
```

## Introduction

Issue 6 built the ESMFold2 folding pipeline; Issue 6a settled the hyperparameters (**68 diffusion steps / 10 refinement loops**). This notebook takes the full **90pid centroid** set — 18,172,960 catalytic-core sequences — and attempts to benchmarking folding on the Compute Canada / Vector **Killarney** cluster (L40S GPUs, `/project` with ~740 GB disk space free).

Rather than reuse the AWS-fleet driver from Issue 6, this run uses a fresh, self-contained script (`resources/260806_issue6_purav_foldrun/fold.py`) with no S3/spot machinery: load ESMFold2 once, fold a length-banded shard of the input, write structures locally. Output goes to `/project` first; S3 archival is a separate, later step.

## Objectives

1. Build the ORFid-keyed fold input from the existing catalytic-core FASTA.
2. Run a **100-sequence test on one L40S** to measure folding throughput and per-structure disk.
3. **Extrapolate** GPU-hours and total storage to all 18.17M centroids, and size the fleet.
4. Scale out via a SLURM job array (one length-band shard per GPU) and archive to S3.

**Part II (2026-10-15): settle the final model and hyperparameters before launching.**

5. Re-test the inference stack: the native `esm` 3.4.1 `EsmFold2Model` (fused kernels, bf16, chunked pair ops) against the August Biohub-`transformers` path.
6. Build fixed benchmark sets: a stratified 90pid sample (set B), a leakage-aware PDB ground-truth set, and an AFDB reference arm.
7. Sweep model (Full / Fast) x diffusion steps x refinement loops, with seed replicates to measure the noise floor.
8. Choose the cheapest configuration that is indistinguishable from the reference beyond seed noise, re-project GPU-hours, and lock `submit_fold.sh`.

---

## Materials and Methods

### Step 0: Environment

The petabite venv (Python 3.12, torch + Biohub `esm`/`transformers` forks) is activated by `activate_petabite.sh`, which also sets `PYTHONNOUSERSITE=1` (keeps `~/.local` from shadowing the venv) and loads the cluster modules. The HuggingFace cache lives on `/project` (home is quota-limited).

```bash
source activate_petabite.sh
export HF_HOME=/home/purav/projects/aip-wanglab/purav/hf_cache   # already in ~/.bashrc

# sanity: the ESMFold2 model class imports cleanly
python -c "from transformers.models.esmfold2.modeling_esmfold2 import ESMFold2Model; print('ok')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
```

> **Precision: fp32 required.** ESMFold2 wraps the 6.3B-param ESMC-6B backbone. bf16 *loads* fine (~13 GB peak VRAM) but its **fold forward pass errors on this build**: `RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16` (an uncasted fp32 tensor meets a bf16 weight, every sequence, regardless of length). So we fold in **fp32** (~25 GB weights, fits the 46 GB L40S with room for activations on these short cores). `--dtype fp32` is the default in `fold.py`.


### Step 1: Build the fold input

`make_fold_input.sh` rewrites the catalytic-core FASTA headers so every record is keyed by its ORFid (`>3|7QJM|||||/95-371` → `>orf3`) — that `>orf<id>` becomes the output filename, so structures join back on ORFid. It then prints `seqkit stats`, from which we record **num_seqs** (expect 18,172,960) and **sum_len** (total residues — the key input to the time estimate).

```bash
cd resources/260806_issue6_purav_foldrun
./make_fold_input.sh
# -> data/90pid_catalytic_orfs.orfid.fa
# note the num_seqs and sum_len it prints; plug them into Step 3.
```

> rewriting headers: /home/purav/projects/aip-wanglab/purav/igem-toronto/90pid_catalytic_orfs.fa -> /home/purav/projects/aip-wanglab/purav/igem-toronto/resources/260806_issue6_purav_foldrun/data/90pid_catalytic_orfs.orfid.fa

> === seqkit stats (num_seqs should be 18,172,960; sum_len = total residues) ===

> file    format  type    num_seqs        sum_len min_len avg_len max_len

> /home/purav/projects/aip-wanglab/purav/igem-toronto/resources/260806_issue6_purav_foldrun/data/90pid_catalytic_orfs.orfid.fa

> FASTA   Protein 18172960       4998232825      29      275.0   59275

### Step 2: Test run — 100 sequences on 1 GPU

Grab one L40S, confirm the model loads (`--load-only`), then fold the first 100 records. `--s3` doesn't exist in this script — output is always local.

```bash
salloc --gres=gpu:1 --cpus-per-task=4 --mem=48G --time=1:00:00   # + your --account/--partition

cd resources/260806_issue6_purav_foldrun

# 1) confirm weights resolve + measure load/VRAM (no folding)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/test100 --load-only
```

In [ ]:
# 2) fold 100 (bf16; fall back to --dtype fp32 only if bf16 errors)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/test100 \
    --limit 100 --num-sampling-steps 68 --num-loops 10 --dtype fp32

This writes `data/test100/structures/*.cif`, `data/test100/metrics/*.json`, and `data/test100/progress.shard1of1.csv`. Re-running is a no-op (finished `.cif` files are skipped).

### Step 3: A representative, length-stratified sample

The 100-seq test folded only 29-34aa cores (`fold.py` sorts by length, so `--limit` takes the shortest). Fold cost is super-linear in length, so that throughput is ~20x too optimistic. Added `fold.py --sample`, which picks N sequences evenly spaced across the length range for an unbiased timing curve.

```bash
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/sample --sample 80 --max-len 2048
```


**Estimate.** `estimate_fleet.py` fits `wall = a * L^b` from the sample and integrates it over the true length histogram (throughput x total residues, not mean-time x count).

```bash
python estimate_fleet.py --test-dir data/sample --length-hist data/length_hist.tsv --max-len 2048
```


> **Note.** This first estimate used a provisional 2048aa cap and had not yet measured the real OOM ceiling or tested ESMFold2-Fast. The `calibrate.sbatch` job below did both in one unattended run, and the numbers were refined (see the calibration note).


### Step 4: Test ESMFold2-Fast

`biohub/ESMFold2-Fast` is not MSA-conditioned and has a lighter 24-layer trunk (vs 48). It relies directly on the protein language model embedding and reportedly matches ESMFold2 with little quality loss, so it should fold faster. Same model class, same fold API, same params, only the repo id changes (`fold.py --fast`).


These manual runs are what `calibrate.sbatch` bundles (stratified sample + ladder OOM probe + estimate + quality diff) into one unattended job, for both models.

```bash
# Fast on the same stratified 80 (seed 0 -> identical sequences as data/sample)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/sample_fast --sample 80 --max-len 1280 --fast

# estimate + quality diff (mean pLDDT / pTM, full vs Fast on the shared seqs)
python estimate_fleet.py --test-dir data/sample_fast --length-hist data/length_hist.tsv --max-len 1280
```


### Calibration results (2026-08-07)

One `calibrate.sbatch` job on one L40S produced all of this: stratified samples for both models, a length-ladder OOM probe, the estimates, and a quality diff.

- **bf16 unusable** on this build (Float/BFloat16 matmul in the fold pass); **fp32 mandatory** (~25 GB).
- **fp32 OOM ceiling ~1300aa** (1285 folds, 1529 OOMs), same for both models (memory = shared ESMC-6B backbone + L^2 pair tensor). Cap `--max-len 1280` keeps 99.75%; OOMs cost ~25 min each, so cap below the ceiling.
- **Fast vs Full** (79 shared seqs): 1.78x faster (median 4.79s vs 8.52s) for -2.05 pLDDT (81.8 vs 83.9) and -0.022 pTM.

**Projection** (foldable <=1280aa: 18,128,069 seqs, 4.92 B residues):

| | Full | **Fast (chosen)** |
|---|---|---|
| GPU-hours | 66,001 | **36,034** |
| @64 / 128 GPUs | ~43 / 21.5 d | **~24 / 12 d** |
| storage | 3.15 TB | 3.15 TB (~5 offload waves) |

**Decision: ESMFold2-Fast** — nearly half the compute for ~2 pLDDT, advantage grows with length. The 44,891 seqs >1280aa (0.25%) are deferred to an H100 run.


> **Revisited 2026-10-15 (Part II).** Three things this calibration did not test: (1) a newer inference stack, since `esm` 3.4.1 ships its own `EsmFold2Model` that the Biohub README recommends for production inference; (2) structural agreement and seed noise, because Full vs Fast was compared on mean pLDDT only; (3) steps / loops on Fast and on the 90pid distribution, since issue 110 tuned them on Full with 121 PDB-backed 225-310 aa ORFs, where global TM saturated at ~0.99. The Fast decision and the fp32 / 1280 aa constraints are therefore re-tested in Steps 7-14 before anything is launched.

### Step 5: Scale out (SLURM job array)

Locked config in `submit_fold.sh`: **`FAST=1`** (ESMFold2-Fast), **`NSHARDS=1000`** (Killarney `MaxArraySize`=1001), `--max-len 1280`, fp32. Each array task folds one **cost-balanced** shard (`select_shard` splits by `L^1.2` cost, not count, so every shard is ~37 GPU-h despite the 15× length skew) on one L40S. The 2-day partition (`gpubase_l40s_b4`) fits a shard in one job → single pass, no resubmits. `#SBATCH --account=aip-wanglab` is already set (matches `boltz_petase`).

```bash
cd resources/260806_issue6_purav_foldrun
TESTN=2 ./submit_fold.sh     # validate plumbing on 2 shards first
squeue --me
tail -f logs/fold90pid-<jobid>_1.out   # expect: loaded (biohub/ESMFold2-Fast) + folds ticking

./submit_fold.sh             # all 1000 shards; set MAXCONCURRENT to your allocation's GPU limit
```

**Resume is free**: a killed/preempted shard re-runs from where it stopped (finished `.cif` files are skipped). Wall-clock ≈ 36,069 GPU-h / MAXCONCURRENT (16→94d, 32→47d, 64→24d).

**Storage: ~5 waves.** 3.15 TB won't fit the 740 GB budget, so archive and clear per wave:
```bash
./sync_to_s3.sh          # dry-run
./sync_to_s3.sh --go     # upload structures/ + metrics/, then delete the local wave
```

**Long tail.** The 44,891 seqs >1280aa (0.25%) are skipped on L40S; fold them later on an H100 partition (80 GB fits the L² tensor).

OUT: `resources/260806_issue6_purav_foldrun/data/fold90pid/{structures,metrics}/`

> **Not launched.** As of 2026-10-15 no shard of this array has run (no `data/fold90pid/`, nothing on `s3://.../esmfold2-centroids/90pid/` beyond one test upload). The model, steps, loops, precision, length cap and partition in `submit_fold.sh` are placeholders until Step 15 locks them from the Part II benchmark.

### Step 6: Analysis (stub)

Downstream quality analysis (pLDDT/pTM distributions, comparison against experimental structures, FoldSeek novelty) to be added once a representative batch has folded.

---

## Part II: Final model and hyperparameter benchmark (2026-10-15)

### Why revisit

The August decision (ESMFold2-Fast, fp32, 68 steps / 10 loops, 1280 aa cap, ~36,000 GPU-h) rested on evidence with gaps:

1. **Steps / loops were only swept on Full**, on 121 PDB-backed ORFs of 225-310 aa with mean pLDDT ~95. Every config scored ~0.99 TM, so the metric could not separate them.
2. **Fast vs Full was compared on mean pLDDT only** (79 seqs, one seed). Structural agreement and run-to-run noise were never measured.
3. **No seed replicates**, so a -0.15 pLDDT difference had no noise floor to compare against.
4. **The inference stack moved.** `esm` 3.4.1 (2026-09-16) has a native `EsmFold2Model` with fused kernels, bf16 trunk autocast and chunked L^2 ops, which could change cost, the bf16 failure and the length cap.
5. **The 90pid set is not like the PDB set.** 90% of centroids are Logan ORFs called stop-to-stop: only 2.4% of Logan centroids start with Met and 52% have no stop (contig-truncated). A quarter of centroids are under 118 aa. Hyperparameters can matter there and not on solved hydrolases.

### Plan and decision rule (fixed before the set-B sweep was scored)

| Phase | Step | Question |
|---|---|---|
| 0 | 7-9 | Does the native stack reproduce the August folds, and how much faster / lighter is it? Which kernel and ESMC precision? New length ceiling? |
| 1 | 10-12 | Fixed benchmark sets: stratified 90pid sample (B), PDB ground truth, AFDB arm |
| 2 | 13 | Full / Fast x {32, 68} steps x {3, 5, 10} loops, reference Full 100 / 20, seeds 1-2 on the 68 / 10 configs |
| 3 | 14-15 | Score, choose, re-project, lock the launch config |

**Decision rule.** A configuration is acceptable if, against the Full 100-step / 20-loop reference on set B, it is no worse than two seeds of the same configuration are against each other on:
- **confidence-loss rate**: fraction of sequences where the reference is confident (pLDDT >= 70) and the configuration drops > 15 pLDDT;
- **median TM to the reference on reference-confident sequences** (pLDDT >= 70). Low-confidence sequences are excluded because seed noise dominates there;
- **fraction with pLDDT < 70** and the **geometry proxy** (issue 111 `cheap_geom`);
- and it stays within seed spread of the reference on **PDB ground truth** (TM and CA-lDDT vs experiment, cluster-weighted).

Among acceptable configurations, pick the one with the **lowest projected GPU-hours** for the 18.17M centroids.

### Step 7: Native `esm` 3.4.1 environment

**Rationale.** The Biohub README (v3.4.1) states the `esm` package is the production inference path ("optimized dependencies and inference path", Fold-CP for long sequences), while the Transformers implementation is for experimentation and "uses a different dependency stack". The installed `~/petabite` venv has `esm` 3.3.0 from `main` @ `26b0bc2` (pre 2026-08-21), whose `esmfold2` package has only the input builder; the model class came from the Transformers fork. `petabite` also installs `esm@main` unpinned for ESMC fine-tuning, so upgrading it in place would change that environment. A **separate venv** keeps the August runs reproducible.

`~/install_esmfold2.sh` follows `install_petabite.sh`: numpy / scipy / pandas / rdkit come from `scipy-stack/2024a` + `rdkit` modules (never pip numpy 2.x, which breaks scipy-stack with "numpy.dtype size changed"), everything else from `--no-index` Compute Canada wheels, `esm` with `--no-deps`. Two exceptions: `torch==2.11.0` (esm 3.4.1 pins `>=2.11,<2.12`) and `zstd==1.5.7.2` from PyPI (no cp312 CC wheel; `~/petabite` took the same). flash-attn is omitted (the CC wheel is built against torch 2.9).

```bash
bash ~/install_esmfold2.sh               # -> ~/esmfold2-341
source activate_esmfold2.sh              # repo root; modules + venv + PYTHONNOUSERSITE + HF_HOME (aip-qmorris)
python -c "from esm.models.esmfold2 import EsmFold2Model, ESMFold2InputBuilder; print('ok')"
```

**`fold.py` additions** (the August path stays the default, `--backend hf`):

| Flag | Meaning |
|---|---|
| `--backend native` | `esm.models.esmfold2.EsmFold2Model.from_pretrained(repo, device=..., esmc_precision=...)` |
| `--esmc-precision bf16\|fp32\|fp8` | ESMC-6B backbone precision (fp8 needs H100 + TransformerEngine) |
| `--kernel none\|fused\|cuequivariance` | `model.set_kernel_backend(...)` for the trunk |
| `--chunk auto\|none\|<int>` | `model.set_chunk_size(...)`; `auto` = no chunking up to `--chunk-above` aa (default 1000), chunk 64 beyond |
| metrics JSON / progress CSV | now record `backend`, `esmc_precision`, `kernel`, `chunk`, `fast` and per-fold `peak_vram_gb` |

### Step 8: Smoke test on one L40S

**Rationale.** Compute nodes may lack internet, and the native loader reads the published HF-layout checkpoints through its own converter. Before committing GPU-hours, confirm it loads `biohub/ESMFold2-Fast` from the cached weights (`HF_HUB_OFFLINE=1`) and folds with each kernel backend.

```bash
cd resources/260806_issue6_purav_foldrun
srun --account=aip-qmorris --partition=gpubase_interac --gpus-per-node=l40s:1 --mem=64G \
     --cpus-per-task=4 --time=0:45:00 ./smoke_native.sh     # 3 x ~260aa, kernel none / fused / cuequivariance
```

OUT: `data/smoke_native_{none,fused,cuequivariance}/`

**Results.** Loads offline from the cached weights and folds with all three kernels. 3 x 252-275 aa, ESMFold2-Fast, ESMC bf16, chunk none:

| kernel | warm fold time (275 aa) | peak VRAM | mean pLDDT (3 seqs) |
|---|---|---|---|
| none | 4.5 s | 14.9 GB | 89.2 / 93.4 / 96.0 |
| fused | 2.5 s | 14.7 GB | 89.1 / 93.4 / 96.0 |
| cuequivariance | 2.7 s | 14.9 GB | 89.2 / 93.4 / 96.0 |

The first fold of each run is slower (kernel compilation). Peak VRAM is ~15 GB versus ~25 GB for the August fp32 weights alone.

### Step 9: Phase 0, inference-stack benchmark

**Rationale.** Re-fold the **identical** 79-sequence length-stratified sample used in August (`--sample 80 --max-len 2048`, seed 0) so every native run pairs one-to-one with `data/sample` (Full, HF, fp32) and `data/sample_fast` (Fast, HF, fp32). This isolates the stack from the model and settles kernel and precision before the larger sweep. Seed-1 replicates give the first noise floor. A length ladder (505-2052 aa) with chunk 64 finds the new memory ceiling for each model.

`compare_runs.py` pairs two fold dirs by ORF and reports TM-score between the two predicted structures (tmtools, CA), RMSD, delta mean pLDDT / pTM, speedup and peak VRAM. It was also run on the August pair (`data/sample` vs `data/sample_fast`) because Full vs Fast structural agreement had never been measured.

```bash
sbatch stack_bench.sbatch        # aip-qmorris, 1 x L40S, 3h
#   fast   bf16  kernel none / fused / cuequivariance   (chunk none)
#   fast   bf16  fused  seed 1                          (noise)
#   fast   ESMC fp32  fused                             (precision)
#   full   bf16  fused, and seed 1                      (noise)
#   ladder fast / full, fused, chunk 64, --max-len 100000

python compare_runs.py data/sample data/sample_fast --out data/cmp_full_vs_fast_hf.tsv     # August Full vs Fast
python compare_runs.py data/sample_fast data/stack/fast_bf16_fused                           # stack, Fast
python compare_runs.py data/sample      data/stack/full_bf16_fused                           # stack, Full
python compare_runs.py data/stack/fast_bf16_fused data/stack/fast_bf16_fused_seed1           # seed noise, Fast
python compare_runs.py data/stack/full_bf16_fused data/stack/full_bf16_fused_seed1           # seed noise, Full
python compare_runs.py data/stack/fast_bf16_fused data/stack/fast_esmcfp32_fused             # ESMC precision
python compare_runs.py data/stack/full_bf16_fused data/stack/fast_bf16_fused                 # Full vs Fast, native
```

OUT: `data/stack/*`, `logs/stack-<jobid>.out`

**Results** (job 6286014, 1 h 14 min, 79 shared sequences, 29-833 aa).

| comparison | total-time speedup | mean d pLDDT | median TM between runs | TM < 0.5 | peak VRAM (833 aa) |
|---|---|---|---|---|---|
| Fast: native fused vs August HF fp32 | **2.77x** | -0.01 | 0.928 | 12 | 23.2 GB |
| Fast: native no-kernel vs August HF | 1.41x | +0.04 | 0.928 | 12 | 25.3 GB |
| Fast: native fused vs native no-kernel | 1.96x | -0.05 | 1.000 | 0 | |
| Fast: ESMC fp32 vs ESMC bf16 (fused) | bf16 1.18x faster | +0.02 | 1.000 | 0 | 36.1 vs 23.2 GB |
| **Full: native fused vs August HF fp32** | **3.58x** | -0.05 | 0.953 | 8 | 25.9 GB |
| Fast seed 0 vs seed 1 (native) | | -0.25 | 0.919 | 12 | |
| Full seed 0 vs seed 1 (native) | | -0.08 | 0.956 | 8 | |
| Full vs Fast (native) | Fast 1.46x | **-2.01** (worst -45.0) | 0.952 | 7 | |
| Full vs Fast (August HF) | Fast 1.89x | -2.05 (worst -42.8) | 0.928 | 8 | |

- **Kernel: fused.** Fastest, and structures match no-kernel (TM 1.000). **cuequivariance is dropped**: it crashed on the 833 aa sequence (`TypeError: attention_pair_bias() got an unexpected keyword argument 's'`, an esm 3.4.1 / cuequivariance 0.13 mismatch).
- **ESMC precision: bf16.** fp32 gives identical structures and confidence, is 18% slower and uses 13 GB more. The August bf16 failure was a Transformers-path bug.
- **Structural disagreement between configs is mostly diffusion noise.** Fast against itself with another seed disagrees as much as Fast against Full, binned by Full pLDDT:

| Full pLDDT band | n | median TM, Fast seed 0 vs 1 | median TM, Full vs Fast | TM < 0.5 (seed / model) |
|---|---|---|---|---|
| >= 90 | 45 | 0.986 | 0.989 | 0 / 0 |
| 70-90 | 17 | 0.792 | 0.785 | 2 / 3 |
| < 70 | 17 | 0.461 | 0.554 | 10 / 5 |

- **But Fast's confidence collapses are real.** On the three short sequences where Full was confident and Fast was not, Fast reproduces the collapse on both seeds and both stacks: `orf759477947` (120 aa) Full 84.8 vs Fast 48.4 / 48.1 / 48.7; `orf1318452266` (103 aa) 89.9 vs 47.1 / 45.8 / 48.8; `orf100574222` (87 aa) 73.3 vs 45.9 / 46.5 / 46.3. This motivated the confidence-loss metric in Step 14.
- **Length ladder** (fused, chunk 64, no allocator setting): Fast folds 1529 aa and OOMs from 1537 aa; Full folds 1284 aa and OOMs from 1529 aa. OOMs now fail in seconds, not the ~25 min stalls of August. With `expandable_segments:True` the sweep went further (Step 13).

### Step 10: Benchmark set B (stratified 90pid sample)

**Rationale.** The sweep needs sequences that look like the 18.17M fold targets, including the hard cases the PDB set cannot represent. `make_bench_set.py` samples `centroids90.parquet` (seeded, reproducible) in cells of length bin x origin class:
- **8 length bins**: 29-99, 100-149, 150-249, 250-349, 350-499, 500-799, 800-1280, 1281-2048 aa. Short bins are over-represented relative to their share because that is where Fast lost confidence in Step 9; 1281-2048 tests the new length ceiling.
- **3 origin classes**: NR, Logan complete (has stop), Logan truncated (no stop, cut at the contig edge).
- 20 per cell (480), plus **100 AFDB-backed centroids** (for Step 12) and **40 plasticome / PAZy centroids** (known plastic-degrading enzymes).

Sampling runs on the small columns first and joins sequences only for chosen rows. Loading all 18M sequences exceeds the login-node memory cap, so it runs as a CPU job.

```bash
srun --account=aip-qmorris --cpus-per-task=8 --mem=32G --time=0:30:00 \
     bash -c "source ../../activate_esmfold2.sh; python make_bench_set.py \
              --parquet /project/6122149/purav/petadex-corpus/centroids90/centroids90.parquet"
```

OUT: `data/bench/bench_B.fa`, `data/bench/bench_B_map.tsv` (orf id, origin class, length bin, arm, AFDB key, HMM, n_members)

**Results.** 620 sequences, 307,088 residues.

| arm | origin class | n |
|---|---|---|
| strat | NR | 160 |
| strat | Logan complete | 160 |
| strat | Logan truncated | 160 |
| afdb | NR | 100 |
| plasticome | NR / PAZy | 25 / 15 |

By length bin: 29-99 (60), 100-149 (60), 150-249 (67), 250-349 (124), 350-499 (98), 500-799 (88), 800-1280 (63), 1281-2048 (60).

### Step 11: PDB ground-truth set

**Rationale.** Rebuild issue 110's cleanest class ("direct": the ORF's own accession is a PDB chain, so the experimental structure is of that exact sequence) from public data, wider than 225-310 aa and without needing S3 access.
1. `gt_scan_pdb_accessions.py` scans the HF `orfs/` spine remotely (7 columns only, never the 28 GB sequence column) for NR / PAZy accessions matching a PDB id (`7QJM`, `5YNS_A`).
2. `gt_build_set.py` queries RCSB for release date, method and resolution; keeps **X-ray or EM at <= 3.0 A** and <= 2048 aa; labels entries released after the ESMFold2 training cutoff used in issue 110 (**2021-09-30**) as `clean`, and adds **40 length-matched pre-cutoff `control`s** (leakage ceiling: if controls do not score high, the scoring is wrong); downloads the mmCIFs from RCSB; pulls the ORF sequences from the spine.

Known limits: many clean entries are mutants of one enzyme, so summaries are also **cluster-weighted** (each 90% cluster counts once); and the set has no very short sequences.

```bash
srun --account=aip-qmorris --cpus-per-task=8 --mem=24G --time=1:00:00 \
     bash -c "source ../../activate_esmfold2.sh; python gt_scan_pdb_accessions.py"   # -> data/gt/pdb_accession_orfs.tsv
srun --account=aip-qmorris --cpus-per-task=8 --mem=24G --time=1:00:00 \
     bash -c "source ../../activate_esmfold2.sh; python gt_build_set.py"             # -> data/gt/{gt.fa,gt_map.tsv,pdb/}
```

OUT: `data/gt/gt.fa`, `data/gt/gt_map.tsv`, `data/gt/pdb/*.cif`

**Results.** 1,448 ORFs in the corpus carry a PDB accession (1,444 NR, 4 PAZy; 1,430 distinct entries, 109 of them 90% centroids). RCSB returned metadata for 1,427. After filtering: **204 clean** (post-2021-09-30; 200 PDB entries, 118 distinct 90% clusters, 190-1026 aa, median 298) and **40 controls** (224-627 aa, median 330, 34 clusters). All mmCIFs downloaded; 244 sequences in `gt.fa`.

### Step 12: AlphaFold DB reference arm

**Rationale.** 132,010 centroids have their own AFDB structure (`centroid_afdb_s3_key`). The 100 AFDB-backed centroids in set B (plus NR stratum members that also have one) give a paired ESMFold2 vs AlphaFold2 comparison on our own sequences. AFDB derives from UniProt and these proteins are likely in ESMFold2's training data, so this measures **consistency with AF2, not accuracy on novel sequences**. It complements the PDB set rather than replacing it.

The `~/petabite` AWS CLI is v1, which has no `aws sso login`, so AWS CLI v2 was installed to `~/aws-cli-v2` (official installer, separate from the venv).

```bash
~/aws-cli-v2/bin/aws sso login --profile igem-purav --use-device-code
# keys from bench_B_map.tsv -> data/afdb/afdb_keys.txt, then per centroid:
~/aws-cli-v2/bin/aws --profile igem-purav s3 cp <centroid_afdb_s3_key> data/afdb/orf<id>.cif
# scored with score_gt.py using data/afdb_gt/{gt_map.tsv, pdb -> ../afdb}
```

OUT: `data/afdb/*.cif`, `data/afdb_gt/gt_map.tsv`

**Results.** All 110 AFDB models downloaded (100 from the AFDB arm plus 10 NR stratum members with their own AFDB entry). Lengths: 150-249 (5), 250-349 (40), 350-499 (28), 500-799 (32), 800-1280 (5).

### Step 13: Phase 2, model x steps x loops sweep

**Rationale.** Hold the stack fixed at the Step 9 choice (native, ESMC bf16, fused kernel, `--chunk auto`) and vary only the folding hyperparameters. Steps {32, 68} bracket the paper default (68; "no benefit of increased sampling steps") and the cheap end issue 110 tested. Loops {3, 5, 10} extend below issue 110's lower bound of 5. The **reference is Full, 100 steps, 20 loops**, the highest-compute setting in issue 110. **Seeds 1 and 2** on the Full and Fast 68 / 10 configs give the noise floor that every comparison is judged against. The same 17 configs fold both set B and the PDB set.

| task | config | model | steps | loops | seed |
|---|---|---|---|---|---|
| 1 | ref_full_s100_l20 | Full | 100 | 20 | 0 |
| 2-7 | full_s{68,32}_l{10,5,3} | Full | 68 / 32 | 10 / 5 / 3 | 0 |
| 8-13 | fast_s{68,32}_l{10,5,3} | Fast | 68 / 32 | 10 / 5 / 3 | 0 |
| 14-15 | full_s68_l10_seed{1,2} | Full | 68 | 10 | 1, 2 |
| 16-17 | fast_s68_l10_seed{1,2} | Fast | 68 | 10 | 1, 2 |

```bash
sbatch --array=1-17%8 sweep.sbatch                                                 # set B -> data/sweep/<config>/
INPUT=data/gt/gt.fa OUTROOT=data/sweep_gt sbatch --export=ALL,INPUT,OUTROOT \
      --job-name=fold-sweep-gt --array=1-17%8 sweep.sbatch                         # PDB set -> data/sweep_gt/<config>/
```

Every task uses `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, `HF_HUB_OFFLINE=1`, `--max-len 2048`. Sequences above a model's memory ceiling fail fast and are logged; configs are compared only where the reference folded.

**Scheduling notes.** Killarney assigns the L40S partition from the requested walltime (b1 <= 3 h, 168 nodes; b2 <= 12 h, 126 nodes). `aip-qmorris` has no GPU or job cap, and its fairshare (0.58) is well above `aip-wanglab` (0.11). The set-B array was submitted at 12 h (b2) with an 8-task throttle. The throttle was raised mid-run and the last five tasks moved to b1:

```bash
scontrol update JobId=<array> ArrayTaskThrottle=16
scontrol update JobId=<array>_<task> TimeLimit=03:00:00 Partition=gpubase_l40s_b1
```

OUT: `data/sweep/<config>/`, `data/sweep_gt/<config>/`, `logs/sweep-<jobid>_<task>.out`

**Results.** Set B: array 6286714 (16 tasks) plus 6290094 (task 14 resubmitted). Task 14 first landed on kn161, which raised `cudaErrorECCUncorrectable`, and was reported `COMPLETED` because the `grep` pipe masked the exit code. `sweep.sbatch` now sets `pipefail`, and the rerun excluded kn161. PDB set: array 6286952, all 17 configs x 244 folded, no failures.

With `expandable_segments:True` the realised L40S ceilings are **Full 1410 aa** (first OOM 1425 aa, peak 46.4 GB) and **Fast 1754 aa** (first OOM 1768 aa). Full configs folded 578 / 620 and Fast 612 / 620; every failure is an OOM above the ceiling. The reference folded through 1305 aa when scored. The Full seed replicates were still finishing their last long sequences, so 549 and 559 of the shared ORFs were scored.

### Step 14: Scoring

**Rationale.** Score each configuration on three independent references and against the seed-noise floor.

- **`score_sweep.py` (set B vs the Full 100 / 20 reference).** Per ORF: TM to the reference structure, delta pLDDT / pTM, geometry proxy from issue 111 (`cheap_geom.score_structure`: clashes per 1k atoms, % Ramachandran favoured), wall time and peak VRAM. Per config: median TM on reference-confident sequences, fraction TM < 0.5, **confidence-loss rate**, fraction pLDDT < 70, geometry, per-length-bin TM, and **projected GPU-hours** (power-law fit of wall time vs length, integrated over `data/length_hist.tsv`). Seed replicates are scored against their seed-0 run with the same metrics.
- **`score_gt.py` (vs experiment).** TM normalised by the experimental chain (TM-align is sequence-independent, so a full-ORF prediction against a partial PDB chain is handled), CA-lDDT over the TM-align correspondence (inclusion 15 A; thresholds 0.5 / 1 / 2 / 4 A), RMSD, per role (clean / control) and cluster-weighted.
- **`score_gt.py` on the AFDB arm** (`--gt data/afdb_gt`), same metrics with AFDB models as reference.

Scoring runs as CPU jobs (process pool), not on the login node.

```bash
J="srun --account=aip-qmorris --cpus-per-task=16 --mem=32G --time=0:45:00"
$J bash -c "source ../../activate_esmfold2.sh; python score_sweep.py --sweep-dir data/sweep --ref ref_full_s100_l20 \
            --map data/bench/bench_B_map.tsv --length-hist data/length_hist.tsv --out data/sweep_scores --workers 16"
$J bash -c "source ../../activate_esmfold2.sh; python score_gt.py --sweep-dir data/sweep_gt --gt data/gt      --out data/sweep_gt_scores   --workers 16"
$J bash -c "source ../../activate_esmfold2.sh; python score_gt.py --sweep-dir data/sweep    --gt data/afdb_gt --out data/sweep_afdb_scores --workers 16"
```

OUT: `data/sweep_scores/{summary,per_orf}.tsv`, `data/sweep_gt_scores/{summary,per_orf}.tsv`, `data/sweep_afdb_scores/{summary,per_orf}.tsv`

Not done: catalytic-triad RMSD. Residue positions exist only in the `catalytic_domains_v260701` layer (2.15% of ORFs), so it is deferred.

**Results.** See Part II under Results & Discussion.

### Step 15: Lock the configuration and launch plan

**Decision (Purav, 2026-10-15): hybrid, and fold the 60% centroids first.**
- **Hybrid model.** Full below 150 aa (where every Fast config lost confidence) and Fast from 150 aa (zero confidence loss in 441 reference-confident sequences, and equal to Full on PDB and AFDB). Both at **32 diffusion steps / 3 refinement loops**, native stack, ESMC bf16, fused kernel.
- **60% centroids first.** All 1,814,296 60% centroids are also 90% centroids (checked: 1,814,296 / 1,814,296 present in `centroids90.parquet`), so this run is the first tenth of the 90% run and nothing is redone when extending. It is ~840 GPU-h of folding (about half a day on 64 L40S), fits the 740 GB disk budget without offload waves, and tests real throughput and the filesystem before the 18M run.

| level | centroids | hybrid GPU-h | all-Full GPU-h | days at 64 / 128 L40S (hybrid) | storage (plain mmCIF) | > 1740 aa |
|---|---|---|---|---|---|---|
| 60% | 1,814,296 | ~840 | ~1,700 | 0.5 / 0.3 | ~400 GB | 3,440 |
| 90% | 18,172,960 | ~7,280 | ~13,300 | 4.7 / 2.4 | ~3.2 TB | 15,858 |

GPU-h are fold time only, from the sweep's per-tier power-law fits (Full 32/3 under 150 aa: `wall = 0.502 * L^0.159`, ~1.0 s at 100 aa; Fast 32/3 from 150 aa: `wall = 8.99e-5 * L^1.655`, 0.9 s at 264 aa, 8.3 s at 1000 aa, 21 s at 1740 aa), +/-10% node variation. Budget ~80% efficiency for model load, file writes and queue gaps.

**Input** (`make_fold_input_hf.py`). Centroid IDs from `petadex/petadex-corpus-full` `dims/clusters60.parquet` (`centroid_orf_id`); sequences from `petadex/petadex-corpus-centroids90` (`centroid_sequence`, uppercased, trailing `*` stripped). Skips centroids with their own AFDB structure (pass `--include-afdb` to fold them anyway) and exact duplicate sequences (mapped to the folded copy in `manifest.tsv`, column `folded_as`).

```bash
srun --account=aip-qmorris --cpus-per-task=8 --mem=32G --time=0:30:00 \
     bash -c "source ../../activate_esmfold2.sh; python make_fold_input_hf.py --level 60 --outdir data/fold60pid_input"
```

| tier | model | length | to fold | skipped (AFDB / duplicate) | residues |
|---|---|---|---|---|---|
| short_full | Full 32/3 | 40-149 aa | 123,940 | 7 / 0 | 12.2 M |
| mid_fast | Fast 32/3 | 150-1740 aa | 1,667,455 | 19,423 / 31 | 602.4 M |
| tail_h100 | H100, TBD | 1741-59,275 aa | 3,440 | 0 / 0 | 8.9 M |

OUT: `data/fold60pid_input/{short_full,mid_fast,tail_h100}.fa`, `manifest.tsv` (every 60% centroid: tier, skip reason, `folded_as`, c90/c60/c30 ids, AFDB key)

**`fold.py` production options** (defaults leave the benchmark runs unchanged):
- `--gzip`: writes `structures/<id>.cif.gz` (~4x smaller).
- `--subdirs 1000`: outputs under `structures/<NNN>/` and `metrics/<NNN>/`, with NNN = orf number mod 1000, so no directory holds millions of files.
- `--cost-exp` / `--cost-floor`: shard balancing per tier (Full 0.16 / 0, flat; Fast 1.66 / 1000).
- Atomic writes: the structure goes to `.tmp` and is renamed only after the metrics JSON is written, so a job killed mid-write never leaves a truncated file that resume would skip.

**Shards.** 3 h walltime, so `gpubase_l40s_b1` (168 nodes x 4 L40S). About 2.5 h of folding per shard at 80% efficiency gives **20 short_full shards and 400 mid_fast shards** (420 tasks, ~1,030 GPU-h budgeted). A shard cut off at 3 h resumes on resubmission. `MAXCONCURRENT` defaults to 128.

```bash
./submit_fold60.sh                     # dry run: prints both sbatch commands, submits nothing
TESTN=2 ./submit_fold60.sh --go        # plumbing test: shards 1-2 of each tier
./submit_fold60.sh --go                # full launch
sbatch --array=1-1 fold_tail_h100.sbatch   # long-tail trial on one H100 (untested; > ~4000 aa likely needs Fold-CP)
```

OUT: `data/fold60pid/{short_full,mid_fast,tail_h100}/{structures,metrics}/<NNN>/`, `progress.shard<K>of<N>.csv`, `logs/fold60-*.out`

**Plumbing test (2026-10-15 23:01).** `TESTN=2 ./submit_fold60.sh --go` (jobs 6293833 short_full, 6293834 mid_fast). After 17 min: 4,494 folds, 0 failures; fold time matches the sweep fits (Full 1.01 s at 40-62 aa, Fast 0.78 s at 150-162 aa); **end-to-end efficiency 85-92%** including model load (vs 80% budgeted), so the 60% run is ~950 GPU-h. gzip mmCIF + metrics JSON = **167 B/residue** (vs 641 B plain), so the 60% run is ~104 GB and the 90% set ~0.83 TB. Structures parse with all residues; metrics record the right model / steps / loops / kernel.

**File-count limit for the 90% run.** `/project` (aip-qmorris) allows 10 M files. The 60% run writes ~3.6 M (`.cif.gz` + `.json` per sequence), which fits; the 90% run would write ~36 M. Before it: drop per-sequence JSON (the per-shard progress CSV holds the same metrics) and pack each finished shard into one archive.

**Launch (2026-10-15 23:27).** Cluster state at launch: ~2 of 464 schedulable L40S free, 2,329 jobs pending on priority, aip-qmorris fairshare 0.57. Cap of 64 concurrent GPUs (~14% of the pool); expected 15 h at an average of 64, ~30 h at 32.

```bash
FROM=3 MAXCONCURRENT=64 ./submit_fold60.sh --go    # shards 3-20 (6294470) and 3-400 (6294471); test shards 1-2 kept running
scontrol update JobId=6294471 ArrayTaskThrottle=42 # throttle is per array: 18 short + 4 test + 42 = 64 total
# raised back to 64 once the short_full and test shards finished
```

**Structure QC (2026-10-15, ~25 min after launch).** `qc_structures.py` sampled 300 structures per tier across all running shards (13,677 written by then):

```bash
srun --account=aip-qmorris --cpus-per-task=16 --mem=32G --time=0:45:00 \
     bash -c "source ../../activate_esmfold2.sh; python qc_structures.py --run data/fold60pid --input data/fold60pid_input --per-tier 300"
```

| check | short_full (Full) | mid_fast (Fast) |
|---|---|---|
| gzip + mmCIF parse | 300 / 300 | 300 / 300 |
| residue sequence == input FASTA | 300 / 300 | 300 / 300 |
| backbone N/CA/C/O present; heavy-atom completeness | all; 1.000 | all; 1.000 |
| CA-CA median / chain breaks | 3.79 A / 0 | 3.84 A / 0 |
| B-factor mean vs metrics JSON pLDDT | <= 0.002 | <= 0.001 |
| model / steps / loops / kernel / ESMC | Full 32/3 fused bf16 | Fast 32/3 fused bf16 |
| Ramachandran favoured (median) | 96.0% | 96.6% |
| mean pLDDT median (p10-p90); < 70 | 52.1 (46.1-88.7); 81% | 92.4 (84.2-96.1); 1.3% |

Every structure has its metrics JSON; no empty files, no progress failures. The only `.tmp` files belonged to folds being written at that moment. **Reproducibility:** the one 60% centroid also in set B (`orf327985183`) matches the sweep's `full_s32_l3` structure at TM 0.9993, RMSD 0.07 A, pLDDT 42.24 vs 42.27.

The low short-tier confidence comes from the inputs. The 60% short tier is 99.8% Logan (123,740 of 123,940), mostly stop-to-stop fragments. On set B under 150 aa the Full 100/20 reference gives Logan centroids a median pLDDT of 58-63 against 87 for NR, and Full 32/3 matches the reference there (81.1 vs 81.4 over the mixed bin). The first shards are also the shortest lengths. Downstream use should filter or weight by pLDDT.

**Hard length cap: 2,500 aa (decision, Purav, 2026-10-16).** Sequences longer than 2,500 aa are **not folded**, at any centroid level.
- *Why:* pair-representation memory grows as L^2, and above ~2,500 aa ESMFold2 needs Fold-CP across several GPUs. Catalytic ORFs this long are mostly multi-domain fusions or mis-assemblies (the longest centroid is 59,275 aa), and full-length predictions at that size would be low-confidence. The cap removes 1,054 of the 1,814,296 60% centroids (0.06%) and about 4,575 of the 90% centroids (0.025%).
- *Recorded, not dropped:* the 60% list is in `data/fold60pid_input/over_cap_2500.tsv`; `make_fold_input_hf.py --hard-cap 2500` (now the default) marks them `skip_reason=hard_cap` in `manifest.tsv` for later runs. A catalytic-domain crop (domain coordinates from the `catalytic_domains` layer, plus a flank) remains an option for them.
- *ESMC context:* ESMC uses rotary position encodings with a dynamic cache, so 2,049-2,500 aa runs without truncation; it is beyond the typical training crop, and per-structure pLDDT records any quality cost.

**Long tail on H100 (1,741-2,500 aa).** 2,386 60% centroids are above the realised L40S ceiling for Fast (~1,754 aa) and at or below the cap. They fold full-length on H100 (80 GB) with Fast 32 / 3, chunk 32, in 6 shards of ~2 h (`fold_tail_h100.sbatch`, input `data/fold60pid_input/tail_h100_le2500.fa`). Shard 6 (the longest, up to 2,500 aa) runs alone first as the memory test:

```bash
sbatch --array=6   --export=ALL,NSHARDS=6 fold_tail_h100.sbatch   # memory test (job 6316909)
sbatch --array=1-5 --export=ALL,NSHARDS=6 fold_tail_h100.sbatch   # after the test fits
```

**Results (60% run complete, 2026-10-16 15:36).** Launched 2026-10-15 23:27; L40S tiers finished ~14:00 (about 14.5 h wall-clock), H100 tail at 15:36. Verified by `fold60_final_check.sbatch` (`data/fold60pid/qc/final_check_all.txt`): every input sequence has an ok progress row, a `.cif.gz` and a metrics `.json`.

| tier | model | length | folded / input | hardware | GPU-h |
|---|---|---|---|---|---|
| short_full | Full 32/3 | 40-149 aa | **123,940 / 123,940** | L40S | 32.9 |
| mid_fast | Fast 32/3 | 150-1,740 aa | **1,667,455 / 1,667,455** | L40S | 806.3 (793.8 completed + 12.0 timed-out-then-resumed + 0.5 failed) |
| tail_h100 | Fast 32/3, chunk 32 | 1,741-2,500 aa | **2,386 / 2,386** | H100 | 12.3 |
| **total** | | | **1,793,781 structures** | | **851.6** |

Not folded, by design (all in `manifest.tsv` / `over_cap_2500.tsv`): 19,430 with their own AFDB structure, 31 exact-duplicate sequences, 1,054 over the 2,500 aa cap; 1,793,781 + 19,430 + 31 + 1,054 = 1,814,296, every 60% centroid accounted for.

- **Cost.** 851.6 GPU-h against ~950 projected for the 60% set (end-to-end efficiency at or above the 85-92% measured in the plumbing test). Peak 80 GPUs, 64 most of the night.
- **Failures.** 0 sequence-level fold failures. Five L40S nodes went bad during the run: kn140, kn088 and kn112 had a GPU visible to `nvidia-smi` but unusable by CUDA; kn161 and kn117 raised uncorrectable ECC GPU memory errors (see Step 16). 59 shard jobs failed within seconds on them (0.5 GPU-h, nothing folded), and all were resubmitted elsewhere. 4 short-sequence Fast shards (1-4) hit the 3 h limit, because the cost model underestimated per-fold overhead at 150-200 aa; they were resumed. 28 duplicate shard jobs, created by a watcher bug, were cancelled while still pending, so no shard ever ran twice.
- **H100 memory at the cap.** The 2,500 aa sequences peaked at 80.9 GB (of 85.5 GB) with chunk 32, so the cap fits one H100 without Fold-CP.
- **Disk.** 93.7 GB (2.0 + 91 + 0.7 GB gzip mmCIF + metrics), 4.11 M of the 10 M file quota.
- **QC** (`qc_structures.py`, 1,000 per tier). short_full and mid_fast: 0 issues in 2,000; every structure matches its input sequence, with full heavy-atom completeness, CA-CA 3.81 / 3.84 A and no gaps. tail_h100: 81 of 1,000 flagged, all false positives. 80 were CA-CA stretches of 4.4-5.3 A (no gap > 6 A) in low-confidence loops (local pLDDT ~50); 1 was an input ambiguity code (`Z`) written as UNK. The QC now reports gaps > 6 A as breaks and 4.2-6 A as stretched, and compares ambiguity codes as X.
- **Reproducibility.** 80 production structures overlap the Phase 2 sweep (same config and seed): median TM 1.000, minimum 0.964, pLDDT within 1.0 (bf16 run-to-run variation).
- **Confidence** (mean pLDDT, QC sample): short_full median 67.2, 50.8% < 70 (mostly Logan fragments); mid_fast median 92.2, 1.0% < 70; tail_h100 median 82.7, 8.2% < 70.

**For the 90% run.** Reuse the same pipeline. Fix two things first: (1) the file quota (about 36 M files at 90%, so drop per-sequence JSON and pack finished shards), and (2) shard balancing (use the measured per-fold times instead of the power-law fit, so the short-sequence Fast shards fit in 3 h). Start the bad-node watcher from the same login node as the session that monitors it.

### Step 16: Run record, 60% centroid fold (2026-10-15 23:01 to 2026-10-16 15:37)

**Rationale.** A complete record of what the run cost and how it behaved, to plan the 90% run (~10x larger) and to cite in the methods. Every number below comes from SLURM accounting (`sacct`, all 517 `fold60-*` jobs including tests, retries, failures and timeouts) and the 1.79 M rows of per-fold timing in the shard progress CSVs. Nothing is estimated. Regenerate with:

```bash
srun --account=aip-qmorris --cpus-per-task=4 --mem=24G --time=0:45:00 \
     bash -c "source ../../activate_esmfold2.sh; python fold60_run_record.py"
# -> data/fold60pid/qc/run_record.md, run_concurrency.tsv, run_length_profile.tsv
```

**Setup recap.** Native `esm` 3.4.1 ESMFold2 (`~/esmfold2-341`), ESMC-6B bf16, fused kernels, 32 diffusion steps / 3 refinement loops, seed 0, gzip mmCIF + metrics JSON in 1,000 subdirectories. L40S 46 GB nodes on `gpubase_l40s_b1` (3 h jobs) for the Full (< 150 aa) and Fast (150-1,740 aa) tiers; H100 80 GB nodes on `gpubase_h100_b1` for the Fast 1,741-2,500 aa tail (chunk 32). Account `aip-qmorris`, concurrency capped at 64 L40S for most of the run.

**How to read the tables.**
- *Wall-clock.* L40S ran 23:01-14:00, about 15 h; the H100 tail ran 12:46-15:37 (shard 6 from 12:46, shards 1-5 from 13:29), overlapping the end of the L40S run. The full 16 h 36 min is first start to last finish.
- *Queue wait.* The 5 h 18 min median for mid_fast is the array throttle: shards waited for a slot under our own 64-GPU cap, not for the cluster. Full and H100 shards started almost immediately.
- *Run time.* The 0 h 01 m minimum for mid_fast is resumed shards finishing their last few sequences.
- *Fold time vs length.* Below ~250 aa a fold costs a near-constant ~0.8-0.9 s (fixed overhead); beyond that time grows roughly as L^1.66. The power-law shard balancing missed the fixed floor, which is why the four shortest-sequence Fast shards timed out at 3 h.
- *Memory.* There is a ~14 GB floor (bf16 weights + ESMC). Memory grows roughly with L^2: to 46 GB at 1,740 aa (the L40S limit) and 80.9 GB at 2,500 aa on H100, which is why the hard cap sits at 2,500 aa.
- *H100 vs L40S.* At 1,741-1,999 aa an H100 fold takes 14.7 s, against 19.1 s on L40S at the shorter 1,300-1,740 aa, so roughly 1.5-2x faster per residue.
- *Efficiency.* 92-97% of billed GPU time was spent folding; model load, file writes and the 59 bad-node failures cost under 8%.

#### Timeline

| event | time |
|---|---|
| first submission (plumbing test) | 2026-10-15 23:01 |
| first shard started | 2026-10-15 23:01 |
| last short_full shard finished | 2026-10-16 01:31 |
| last mid_fast shard finished | 2026-10-16 14:00 |
| last tail_h100 shard finished | 2026-10-16 15:37 |
| wall-clock, first start to last finish | 16h36m |

#### GPU-hours and job outcomes

| tier | hardware | state | jobs | GPU-h |
|---|---|---|---|---|
| mid_fast | L40S | CANCELLED | 23 | 0.0 |
| mid_fast | L40S | COMPLETED | 400 | 793.8 |
| mid_fast | L40S | FAILED | 54 | 0.5 |
| mid_fast | L40S | TIMEOUT | 4 | 12.0 |
| short_full | L40S | CANCELLED | 5 | 0.0 |
| short_full | L40S | COMPLETED | 20 | 32.9 |
| short_full | L40S | FAILED | 5 | 0.0 |
| tail_h100 | H100 | COMPLETED | 6 | 12.3 |
| **total** | | | **517** | **851.6** |

L40S 839.2 GPU-h, H100 12.3 GPU-h.

#### Shard run time and queue wait (completed jobs)

| tier | completed jobs | run time min / median / max | queue wait median / max |
|---|---|---|---|
| short_full | 20 | 1h33m / 1h38m / 1h47m | 0h00m / 0h01m |
| mid_fast | 400 | 0h01m / 1h55m / 2h53m | 5h18m / 11h40m |
| tail_h100 | 6 | 1h53m / 2h01m / 2h16m | 0h29m / 0h31m |

#### GPUs in use (sampled at the half hour)

| hour | L40S | H100 |
|---|---|---|
| 10-15 23:30 | 20 | 0 |
| 10-16 00:30 | 80 | 0 |
| 10-16 01:30 | 61 | 0 |
| 10-16 02:30 | 70 | 0 |
| 10-16 03:30 | 68 | 0 |
| 10-16 04:30 | 58 | 0 |
| 10-16 05:30 | 62 | 0 |
| 10-16 06:30 | 65 | 0 |
| 10-16 07:30 | 64 | 0 |
| 10-16 08:30 | 64 | 0 |
| 10-16 09:30 | 65 | 0 |
| 10-16 10:30 | 73 | 0 |
| 10-16 11:30 | 45 | 0 |
| 10-16 12:30 | 20 | 0 |
| 10-16 13:30 | 7 | 6 |
| 10-16 14:30 | 0 | 6 |
| 10-16 15:30 | 0 | 2 |

Peak 80 L40S; 6 H100.

#### Per-fold time and throughput

| tier | folds | fold time median (p10-p90) | residues folded | sum of fold time | folds per GPU-hour |
|---|---|---|---|---|---|
| short_full | 123,940 | 0.90 s (0.89-0.92) | 12.2 M | 31.1 h | 3,769 |
| mid_fast | 1,667,455 | 1.09 s (0.79-2.68) | 602.4 M | 743.8 h | 2,068 |
| tail_h100 | 2,386 | 16.57 s (12.56-25.47) | 4.8 M | 12.0 h | 194 |

**End-to-end efficiency** (sum of fold time / GPU-hours billed, including model load, file writes, failed and timed-out jobs): short_full 94.6%, mid_fast 92.2%, tail_h100 97.3%.

#### Fold time and peak GPU memory by length

| length | tier | folds | fold time median | peak memory median / max |
|---|---|---|---|---|
| 40-99 aa | short_full | 70,845 | 0.90 s | 13.9 / 14.0 GB |
| 100-149 aa | short_full | 53,095 | 0.90 s | 14.1 / 14.1 GB |
| 150-249 aa | mid_fast | 310,059 | 0.79 s | 14.2 / 14.4 GB |
| 250-399 aa | mid_fast | 910,885 | 1.05 s | 14.8 / 15.7 GB |
| 400-599 aa | mid_fast | 324,794 | 2.04 s | 16.6 / 18.5 GB |
| 600-899 aa | mid_fast | 92,554 | 4.00 s | 20.0 / 24.7 GB |
| 900-1299 aa | mid_fast | 23,675 | 9.07 s | 26.2 / 31.7 GB |
| 1300-1740 aa | mid_fast | 5,488 | 19.11 s | 36.1 / 46.1 GB |
| 1741-1999 aa | tail_h100 | 1,303 | 14.67 s | 50.3 / 56.6 GB |
| 2000-2249 aa | tail_h100 | 664 | 19.59 s | 61.5 / 68.0 GB |
| 2250-2500 aa | tail_h100 | 419 | 25.96 s | 73.8 / 80.9 GB |

#### H100 tail, per shard

| shard | job | node | sequences | length | run time | peak memory |
|---|---|---|---|---|---|---|
| 1 | 6317150_1 | kn175 | 493 | 1741-1820 aa | 1h53m | 49.2 GB |
| 2 | 6317150_2 | kn175 | 456 | 1820-1911 aa | 1h56m | 52.9 GB |
| 3 | 6317150_3 | kn175 | 419 | 1911-2022 aa | 1h59m | 57.6 GB |
| 4 | 6317150_4 | kn175 | 378 | 2023-2157 aa | 2h03m | 63.7 GB |
| 5 | 6317150_5 | kn175 | 339 | 2157-2308 aa | 2h08m | 70.9 GB |
| 6 | 6316909_6 | kn173 | 301 | 2308-2500 aa | 2h16m | 80.9 GB |

#### Incidents

| node | failed jobs | first / last failure | median time to fail | cause |
|---|---|---|---|---|
| kn140 | 43 | 10-15 23:27 / 10-15 23:39 | 16 s | CUDA unavailable (GPU visible to nvidia-smi) |
| kn161 | 5 | 10-16 02:21 / 10-16 02:30 | 18 s | uncorrectable ECC GPU memory error (also during the Phase 2 sweep) |
| kn088 | 1 | 10-16 05:30 / 10-16 05:30 | 31 s | CUDA unavailable (GPU visible to nvidia-smi) |
| kn117 | 6 | 10-16 09:28 / 10-16 09:29 | 16 s | uncorrectable ECC GPU memory error |
| kn112 | 4 | 10-16 09:29 / 10-16 09:46 | 18 s | CUDA unavailable (GPU visible to nvidia-smi) |

Other failures: 0. Timeouts: 4 (6293834_1, 6293834_2, 6294471_3, 6295015_4), all resumed. Cancelled: 28 (duplicate shard jobs, cancelled while pending; never ran).

### Step 17: Archive to S3, complete the set, free local storage (2026-10-16 to 10-17)

**Rationale.** The structures belong on S3 next to the AFDB pull, not on `/project`. They took 94 GB and 3.59 M files, and the file count matters for the 90% run. The S3 folder should also be the *whole* 60% set, so a user never has to join against AFDB or the duplicate list to find a structure. That means three additions: the AFDB model for centroids that were skipped because AFDB already has them, a copy for exact duplicates, and one index row per centroid.

**Destination.** `s3://petadex-protein-structures/petadex-v1.1-60pid-centroids-structures/`, with the same tier layout as local plus `afdb/`, `inputs/`, `qc/`, `structure_index.tsv` and a README (`S3_README_fold60pid.md` is the tracked copy).

```bash
sbatch fold60_upload_s3.sbatch      # 1. sync tiers + inputs, verify object counts and bytes per prefix; deletes nothing
sbatch fold60_delete_local.sbatch   # 2. after Purav confirmed: delete local structures/ + metrics/ (keeps progress CSVs, qc/, inputs)
sbatch fold60_complete_s3.sbatch    # 3. AFDB copies, duplicates, structure_index.tsv, README; verify every index entry exists
# the 27 inexact-AFDB centroids, folded with the mid_fast settings:
sbatch --job-name=fold60-afdb-refold --array=1 --time=00:30:00 \
  --export=ALL,TIER=mid_fast,INPUT=$PWD/data/fold60pid_input/afdb_refold.fa,OUTDIR=$PWD/data/fold60pid/mid_fast,NSHARDS=1,SCRIPTDIR=$PWD \
  fold_hybrid.sbatch
```

**Results.**

| step | job | when | outcome |
|---|---|---|---|
| upload | 6323482 | 10-16 16:57-23:33 (6 h 36 min) | 3,588,005 files, 100.5 GB. Counts and bytes match per prefix; 300/300 random files have MD5 = S3 ETag |
| delete local | 6364955 | 10-17 15:23-15:36 | 3.59 M files / 93 GB removed; `data/fold60pid` is now 100 MB |
| fold 27 inexact-AFDB centroids | 6370941 | 10-17, 2 min on 1 L40S | 27/27 ok, mean pLDDT 84.6-97.0 (median 92.9) |
| complete the set | 6371060 | 10-17 18:50-19:49 | VERIFIED: all 3,607,080 files named in `structure_index.tsv` are on S3 |

- **AFDB.** The corpus lists an AFDB model (`centroid_afdb_s3_key`) for 19,430 centroids. Each model's sequence, read from its CA atoms, was checked against the centroid's md5. **19,403 match exactly**, and those models are copied (gzipped) to `afdb/structures/<NNN>/orf<ID>.cif.gz`. **27 do not match:** 16 are the same length with 1-16 substitutions and 11 differ in length, up to 565 aa vs 173 aa. Those AFDB entries are related proteins, not the centroid, so the 27 were folded with ESMFold2-Fast instead (Purav, 2026-10-17). The list is in `qc/afdb_inexact_refolded.tsv` and should be reported to Dennis as an upstream mapping issue.
- **Duplicates.** All 31 now have their own file. 30 are copies of their twin's ESMFold2 structure and metrics; for one (`1040467234`) the twin is an AFDB centroid, so it gets the AFDB model.
- **Final set.** 1,813,242 of 1,814,296 centroids (99.94%) have a structure:

| source | centroids |
|---|---|
| ESMFold2 Full (`short_full`, < 150 aa) | 123,940 |
| ESMFold2-Fast (`mid_fast` 1,667,482 incl. the 27; `tail_h100` 2,386) | 1,669,868 |
| AlphaFold DB v4 (`afdb/`) | 19,403 |
| duplicates (copies) | 31 |
| no structure, over the 2,500 aa cap | 1,054 |

- **Also learned.** Each `aws --profile` call with SSO fetches fresh role credentials (16 s per file vs 5 s). Exporting the credentials once per job (`aws configure export-credentials`) took the AFDB copy from about 3 h to about 15 min.


## Results & Discussion

### Part I: August calibration (2026-08-07)

**Key metrics** (ESMFold2-Fast, fp32, L40S, 68 steps / 10 loops):
- Throughput fit `wall = 0.00812·L^1.202`; median fold 4.79 s; mean pLDDT 81.8, pTM 0.729.
- Disk: 641 B/residue (cif) + 194 B/seq (json).

**Projection** (foldable ≤1280aa: 18,128,069 seqs / 99.75%, 4.92 B residues):
- **36,034 GPU-hours** (~24 days on 64 L40S) and **3.15 TB** total.

**Observations**:
- **bf16 is broken on this build** — loads fine, but the fold pass hits a Float/BFloat16 matmul on every sequence. fp32 is mandatory (~25 GB, fits the 46 GB L40S).
- **fp32 OOM ceiling ≈ 1300aa** (1285 folds, 1529 OOMs), identical for Full and Fast — memory is dominated by the shared ESMC-6B backbone + the L² pair tensor. Cap at 1280 (keeps 99.75%); OOMs cost ~25 min each, so cap below the ceiling.
- **Length is the dominant cost driver**: 29–59,275aa, median 264, and cost ~L^1.2, so a naive first-N test (all 29–34aa) over-estimated throughput ~20×. Sharding is cost-balanced to avoid a 15× straggler.
- **Fast vs Full**: 1.78× faster for −2.05 pLDDT / −0.022 pTM. Chose Fast for the first-pass catalog.
- **Storage doesn't fit** the 740 GB budget → ~5 fold-then-offload waves to S3.

**Follow-up questions**:
- How many concurrent L40S does the `aip-wanglab` allocation allow? (sets wall-clock)
- Fold the 44,891 >1280aa sequences on H100 (80 GB), or leave deferred?
- Final S3 layout, and dedup against ESM Atlas prefolds before or after?

### Part II: Final model and hyperparameter benchmark (2026-10-15)

Scored 2026-10-15 22:00. Reference: Full, 100 steps, 20 loops, native stack.

**Phase 0, inference stack.** The native `esm` 3.4.1 stack with fused kernels and bf16 ESMC is **3.58x faster for Full and 2.77x for Fast** than the August Transformers path, with unchanged confidence (mean d pLDDT <= 0.05). bf16 works; ESMC fp32 adds nothing. cuequivariance crashes on longer inputs. Full is now only 1.46x slower than Fast.

**Benchmark sets.** Set B: 620 stratified centroids. PDB: 204 post-cutoff + 40 control structures, X-ray/EM <= 3 A, 190-1026 aa. AFDB: 110.

**PDB ground truth: all configs are equivalent.**

| | TM vs experiment (median, clean) | CA-lDDT (median) | lDDT, worst 10% |
|---|---|---|---|
| range over all 17 configs | 0.9885-0.9903 | 0.9757-0.9798 | 0.926-0.939 |
| ref_full_s100_l20 | 0.9888 | 0.9762 | 0.927 |
| full_s32_l3 | 0.9886 | 0.9757 | 0.926 |
| fast_s32_l3 | 0.9896 | 0.9797 | 0.935 |
| spread between seeds of one config | +/-0.0001 | +/-0.0009 | +/-0.004 |

Controls score the same as clean entries (e.g. reference 0.9895 / 0.9742), so this is not memorisation. These are well-studied 190-1026 aa hydrolases, too easy to separate configurations. They show cheaper settings do not hurt where structure is known.

**AFDB arm: all configs are equivalent.** TM vs AFDB median 0.975-0.983, CA-lDDT 0.979-0.981 across all 17 configs; reference 0.981 / 0.981, full_s32_l3 0.975 / 0.979, fast_s32_l3 0.981 / 0.980; one fold below TM 0.5 in total (a Full seed run).

**Set B vs the reference and the seed-noise floor** (561 shared ORFs, 506 reference-confident):

| config | GPU-h, 18.17M | median TM, ref-confident | TM < 0.5, ref-confident | confidence loss (>15 pLDDT) | pLDDT < 70 |
|---|---|---|---|---|---|
| ref_full_s100_l20 | 48,826 | | | | 9.8% |
| full_s68_l10 | 29,383 | 0.966 | 0.8% | **0** | 9.8% |
| full_s32_l10 | 26,011 | 0.962 | 1.0% | **0** | 10.0% |
| full_s68_l5 | 19,808 | 0.962 | 1.4% | **0** | 10.0% |
| full_s32_l5 | 16,491 | 0.963 | 1.0% | **0** | 10.0% |
| full_s68_l3 | 15,579 | 0.959 | 1.4% | **0** | 9.8% |
| **full_s32_l3** | **13,442** | **0.960** | **1.0%** | **0** | **9.8%** |
| fast_s68_l10 | 16,700 | 0.953 | 3.0% | 38 (7.5%) | 15.2% |
| fast_s68_l5 | 12,484 | 0.953 | 2.6% | 39 (7.7%) | 15.3% |
| fast_s32_l3 | 7,588 | 0.955 | 3.2% | 39 (7.7%) | 15.0% |
| *noise: Full seed vs seed* | 26,438-26,910 | 0.962-0.963 | 0.8-1.2% | 0 | 9.8-10.2% |
| *noise: Fast seed vs seed* | 16,572-16,941 | 0.955-0.960 | 3.6-4.3% | 0 | 13.6-13.7% |

- **Every Fast configuration fails the confidence-loss criterion**: 38-39 sequences against 0 between Fast seeds. All are under 150 aa: 9 of 10 reference-confident sequences at 29-99 aa and 29 of 55 at 100-149 aa, across NR, Logan complete and Logan truncated, median pLDDT 93.8 -> 59.1. **Above 150 aa Fast loses confidence on 0 of 441.**
- **Every Full configuration passes.** No reference-confident sequence drops even 5 pLDDT. TM to the reference stays within the Full seed-vs-seed range (0.959-0.966 vs 0.962-0.963), including on short sequences (full_s32_l3 0.963 under 150 aa vs seed range 0.954-0.965). Geometry proxy medians 1.79-1.81 for all configs.
- **Above 150 aa, Fast agrees slightly less with the Full reference** (median TM 0.954-0.958, 6-10 of 441 below 0.5, against Full's 0.959-0.966 and 4-7), a gap about the size of Fast's own seed noise. On PDB and AFDB it is as accurate as Full.
- GPU-hour projections integrate a power-law time fit over the real length histogram up to 1290 aa. Seeds of one config differ by ~10% (node-to-node speed), so treat them as +/-10%.

**Decision.** Under the pre-set rule the cheapest acceptable configuration was **A: ESMFold2 Full, 32 diffusion steps, 3 refinement loops** (~13,300 GPU-h for the 90% set). **Chosen instead (Purav, 2026-10-15): B, the hybrid** — Full 32 / 3 below 150 aa, Fast 32 / 3 from 150 aa, native stack, bf16, fused. B never loses confidence (0 of 441 reference-confident sequences >= 150 aa; Full covers every short sequence where Fast failed) and is as accurate as Full against PDB and AFDB. Its one cost is slightly lower structural agreement with the Full reference above 150 aa (median TM 0.954-0.958 vs 0.960-0.966; 6-10 vs 4-7 of 441 below 0.5), about the size of Fast's own seed noise. B was defined after scoring, not under the pre-set rule, and is recorded here as such.

**Updated projection (hybrid).** Fast runs up to its realised L40S ceiling (1740 aa), so the hybrid keeps more on L40S than all-Full does.

| level | centroids on L40S | GPU-h (fold time) | at ~80% efficiency | days at 64 / 128 L40S | storage (plain mmCIF) | > 1740 aa (H100) |
|---|---|---|---|---|---|---|
| **60% (first)** | 1,810,836 | **~840** | ~1,030 | 0.5-0.7 / 0.3-0.4 | ~400 GB | 3,440 |
| 90% | 18,157,102 | ~7,280 | ~9,100 | 4.7-5.9 / 2.4-3.0 | ~3.2 TB (~0.8 TB gzip) | 15,858 |
| *August plan (Fast 68 / 10, HF fp32)* | *18,128,069* | *36,034* | | *23.5 / 11.7* | | |

The 90% hybrid is ~5x cheaper than the August plan. Skipping centroids with their own AFDB structure and exact duplicates removes a further ~0.8%. The 60% centroids are a strict subset of the 90% centroids, so the 60% run carries over.

**Follow-up questions:**
- Real throughput, including overhead and filesystem load, from the `TESTN=2` plumbing test, before the full 60% launch.
- H100 trial for the long tail (3,440 at 60%, 15,858 at 90%); > ~4000 aa likely needs Fold-CP.
- Catalytic-triad RMSD on the `catalytic_domains_v260701` subset (2.15% of ORFs), to check active-site geometry directly.
- After the 60% run: sample structures by origin and length for a manual check, then extend to the 90% set.